In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
from os.path import join

import pandas as pd
import numpy as np

# import missingno as msno

from sklearn.model_selection import GridSearchCV

# from sklearn.model_selection import KFold, cross_val_scor

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report



# 데이터 불러오기

In [2]:
# train_data_path = join('../input', 'train.csv')
# sub_data_path = join('../input', 'test.csv')

# data_dir = os.path.join(os.getenv('HOME'), 'work/kaggle_kakr_housing/data')
data_dir = './data'

# hint : os.getenv를 사용하거나 직접 경로를 작성

train_data_path = os.path.join(data_dir, 'train.csv')
sub_data_path = os.path.join(data_dir, 'test.csv')      # 테스트, 즉 submission 시 사용할 데이터 경로

print(train_data_path)
print(sub_data_path)

./data/train.csv
./data/test.csv


In [3]:
data = pd.read_csv(train_data_path)
sub = pd.read_csv(sub_data_path)
print('train data dim : {}'.format(data.shape))
print('sub data dim : {}'.format(sub.shape))

train data dim : (8693, 14)
sub data dim : (4277, 13)


# 데이터 준비

In [4]:
y = data['Transported']
del data['Transported']

# 데이터 탐색

In [5]:
train_len = len(data)
data = pd.concat((data, sub), axis=0)

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   12970 non-null  object 
 1   HomePlanet    12682 non-null  object 
 2   CryoSleep     12660 non-null  object 
 3   Cabin         12671 non-null  object 
 4   Destination   12696 non-null  object 
 5   Age           12700 non-null  float64
 6   VIP           12674 non-null  object 
 7   RoomService   12707 non-null  float64
 8   FoodCourt     12681 non-null  float64
 9   ShoppingMall  12664 non-null  float64
 10  Spa           12686 non-null  float64
 11  VRDeck        12702 non-null  float64
 12  Name          12676 non-null  object 
dtypes: float64(6), object(7)
memory usage: 1.4+ MB


In [7]:
data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines


In [8]:
data['group'] = data['PassengerId'].apply(lambda x: x.split('_')[0])
data['gnum'] = data['PassengerId'].apply(lambda x: x.split('_')[1])

In [9]:
data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,group,gnum
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,0001,01
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,0002,01
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,0003,01
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,0003,02
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,0004,01


In [10]:
planet = pd.get_dummies(data['HomePlanet'])
planet.head()

,Earth,Europa,Mars
0,False,True,False
1,True,False,False
2,False,True,False
3,False,True,False
4,True,False,False


In [11]:
data = pd.concat([data, planet], axis=1)
data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,group,gnum,Earth,Europa,Mars
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,0001,01,False,True,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,0002,01,True,False,False
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,0003,01,False,True,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,0003,02,False,True,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,0004,01,True,False,False


In [12]:
# data[data.isnull().any(axis=1)].sum()
data.isnull().sum()

PassengerId       0
HomePlanet      288
CryoSleep       310
Cabin           299
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Name            294
group             0
gnum              0
Earth             0
Europa            0
Mars              0
dtype: int64

In [13]:
data['cdeck'] = data['Cabin'].apply(lambda x: x.split('/')[0] if pd.notna(x) else x)
data['cnum'] = data['Cabin'].apply(lambda x: x.split('/')[1] if pd.notna(x) else x)
data['cside'] = data['Cabin'].apply(lambda x: x.split('/')[2] if pd.notna(x) else x)

In [14]:
data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,VRDeck,Name,group,gnum,Earth,Europa,Mars,cdeck,cnum,cside
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,...,0.0,Maham Ofracculy,0001,01,False,True,False,B,0,P
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,...,44.0,Juanna Vines,0002,01,True,False,False,F,0,S
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,...,49.0,Altark Susent,0003,01,False,True,False,A,0,S
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,...,193.0,Solam Susent,0003,02,False,True,False,A,0,S
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,...,2.0,Willy Santantines,0004,01,True,False,False,F,1,S


In [15]:
cdeck = pd.get_dummies(data['cdeck'])
cdeck.head()

,A,B,C,D,E,F,G,T
0,False,True,False,False,False,False,False,False
1,False,False,False,False,False,True,False,False
2,True,False,False,False,False,False,False,False
3,True,False,False,False,False,False,False,False
4,False,False,False,False,False,True,False,False


In [16]:
cside = pd.get_dummies(data['cside'])
cside.head()

,P,S
0,True,False
1,False,True
2,False,True
3,False,True
4,False,True


In [17]:
destination = pd.get_dummies(data['Destination'])
destination.head()

,55 Cancri e,PSO J318.5-22,TRAPPIST-1e
0,False,False,True
1,False,False,True
2,False,False,True
3,False,False,True
4,False,False,True


In [18]:
data = pd.concat([data, cdeck, cside, destination], axis=1)
data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,D,E,F,G,T,P,S,55 Cancri e,PSO J318.5-22,TRAPPIST-1e
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,...,False,False,False,False,False,True,False,False,False,True
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,...,False,False,True,False,False,False,True,False,False,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,...,False,False,False,False,False,False,True,False,False,True
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,...,False,False,False,False,False,False,True,False,False,True
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,...,False,False,True,False,False,False,True,False,False,True


In [19]:
data.drop([	'PassengerId', 'HomePlanet', 'Cabin', 'cdeck', 'cside',  'Destination', 'Name'], axis=1, inplace=True)

# 시각화

In [20]:
sub = data.iloc[train_len:, :]
data = data.iloc[:train_len, :]
print(data.shape, data.shape,)

(8693, 27) (8693, 27)


# 모델링

In [21]:
sub = data.iloc[train_len:, :]
x = data.iloc[:train_len, :]

In [22]:
# trade.dropna(how='all', subset=['수출건수', '수출금액', '수입건수', '수입금액', '무역수지'], inplace=True)
data = pd.concat([x, y], axis=1)
data.dropna(how='any', inplace=True)

In [24]:
y = data['Transported']
del data['Transported']
x = data

In [25]:
x.head()

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,group,gnum,...,D,E,F,G,T,P,S,55 Cancri e,PSO J318.5-22,TRAPPIST-1e
0,False,39.0,False,0.0,0.0,0.0,0.0,0.0,0001,01,...,False,False,False,False,False,True,False,False,False,True
1,False,24.0,False,109.0,9.0,25.0,549.0,44.0,0002,01,...,False,False,True,False,False,False,True,False,False,True
2,False,58.0,True,43.0,3576.0,0.0,6715.0,49.0,0003,01,...,False,False,False,False,False,False,True,False,False,True
3,False,33.0,False,0.0,1283.0,371.0,3329.0,193.0,0003,02,...,False,False,False,False,False,False,True,False,False,True
4,False,16.0,False,303.0,70.0,151.0,565.0,2.0,0004,01,...,False,False,True,False,False,False,True,False,False,True


In [26]:
random_state=25

In [27]:
def get_scores(models, train, y):
    df = {}

    for model in models:
        model_name = model.__class__.__name__

        X_train, X_test, y_train, y_test = train_test_split(train, y, random_state=random_state, test_size=0.2)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # df[model_name] = rmse(y_test, y_pred)
        # score_df = pd.DataFrame(df, index=['RMSE']).T.sort_values('RMSE', ascending=False)
        # print(model_name, score_df)

    return y_test, y_pred

In [28]:
model = DecisionTreeClassifier(random_state=random_state)
model

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,25
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [29]:
cols  = ['RoomService', 'FoodCourt',  'ShoppingMall',  'Spa', 'VRDeck']  

In [30]:
y_test, y_pred = get_scores([model], x, y)

# Accuracy

In [31]:
confusion_matrix(y_test, y_pred)

array([[514, 173],
       [164, 566]])

In [32]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.76      0.75      0.75       687
        True       0.77      0.78      0.77       730

    accuracy                           0.76      1417
   macro avg       0.76      0.76      0.76      1417
weighted avg       0.76      0.76      0.76      1417

